# Ablation Notebook 1 — Pre-train · Retrain Baselines (3/7 Split)

**Experiment:** CMF global-mean μ source ablation — forget-set contamination under a 3/7 split.

**Key difference from the original Notebook 1:**  
The forget-class partition is **3 of 10 CIFAR-10 classes** (classes 0, 1, 2 =  
airplane / automobile / bird), giving a **30 % forget / 70 % retain** split  
instead of the original 10 % / 90 % (1/9) split.

| Stage | Output | Path |
|-------|--------|------|
| **A** | Environment setup & repo clone | — |
| **B** | Configuration | — |
| **C** | Dataset & data-loaders | — |
| **D** | Pre-train (reuse original if available) | `checkpoints/ablation_37/pre_train/<dataset>_<arch>_<tag>.pt` |
| **E** | Retrain baseline on 3/7 retain set | `checkpoints/ablation_37/retrain/<dataset>_<arch>_<tag>/<classes>.pt` |
| **F** | CMF fine-tune encoder | `checkpoints/ablation_37/CMF_FT_RemoveFC/<dataset>_<arch>_<tag>.pt` |
| **G** | Save `ablation_config.json` | `checkpoints/ablation_37/ablation_config.json` |

> **After this notebook finishes:**  
> Go to **Output → Add to Dataset** and publish the checkpoints directory.  
> Attach that dataset to **Ablation Notebook 2**.  
> Set `PREV_RUN_DATASET_DIR` to its mount path to reuse the pre-train checkpoint.

> **Recommended:** GPU T4/P100.  
> Full mode (300-epoch pretrain, 50-epoch retrain for ONE group [0,1,2]) ≈ 2–3 h on T4.

## A. Environment Setup

In [ ]:
import subprocess, sys

def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout:
        print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode

sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, copy, random, argparse, collections, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'

if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## B. Configuration

**3/7 ablation-specific settings are marked with `# <-- ABLATION`.**

| Mode | `TEST_MODE` | What it does | Time |
|------|------------|--------------|------|
| **Test** | `True` | 1% data, 1 epoch | ~1 min (CPU) |
| **Full** | `False` | Full CIFAR-10, 300-epoch pretrain, 50-epoch retrain | ~2–3 h (T4) |

In [ ]:
# ======================================================================
#  FLIP THIS to False for the real experiment run
TEST_MODE     = True
TEST_FRACTION = 0.01
# ======================================================================

# If you already ran the ORIGINAL Notebook 1 and attached its output
# as a Kaggle dataset, set ORIG_CKPT_DIR to its mount path so we can
# REUSE the pre-trained checkpoint instead of re-training from scratch.
# The pre-train is identical whether forget_classes=[0] or [0,1,2] —
# it trains on the FULL dataset.
#
# Example: ORIG_CKPT_DIR = '/kaggle/input/cmf-pretrain-checkpoints'
ORIG_CKPT_DIR = None  # <- set to None to train from scratch

# ── Dataset / architecture ─────────────────────────────────────────────
DATASET   = 'cifar10'
ARCH      = 'resnet18'
IS_VIT    = False
DATA_PATH = '/kaggle/working/data'
SEED      = 1234

# ── 3/7 ablation: ONE forget group of 3 classes  <-- ABLATION
# Classes 0=airplane, 1=automobile, 2=bird.
# Produces forget=15000 samples (30%), retain=35000 samples (70%).
FORGET_CLASSES = [0, 1, 2]   # <-- ABLATION: 3 of 10 classes

# Dataset constants
_TOTAL     = {'cifar10': 50000, 'cifar100': 50000, 'tinyimagenet': 100000}
_PER_CLASS = {'cifar10': 5000,  'cifar100': 500,   'tinyimagenet': 500}

NUM_CLASSES = 10
CLASS_LABEL_NAMES = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

# Pre-training hyperparams (identical to original)
if TEST_MODE:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 1, 8, 1
else:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 300, 128, 50

RETRAIN_EPOCHS = 1 if TEST_MODE else 50

# Checkpoint root — ablation gets its own sub-directory
CKPT_ROOT = '/kaggle/working/checkpoints/ablation_37'
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

_MODE_TAG    = 'test' if TEST_MODE else 'full'
_FORGET_STR  = ','.join(str(c) for c in FORGET_CLASSES)  # '0,1,2'

print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Forget classes : {FORGET_CLASSES}  ({len(FORGET_CLASSES)}/{NUM_CLASSES} = 3/7 split)')
print(f'Forget samples : {len(FORGET_CLASSES)*_PER_CLASS[DATASET]}')
print(f'Retain samples : {_TOTAL[DATASET] - len(FORGET_CLASSES)*_PER_CLASS[DATASET]}')
print(f'Pretrain epochs: {PRETRAIN_EPOCHS}  Retrain epochs: {RETRAIN_EPOCHS}')

## C. Dataset & Data-Loaders

In [ ]:
from utils import get_dataset, get_model, get_retain_forget_partition, test, load_encoder_ckpt_safely
from unlearn import unlear_func
from train import train as train_one_epoch
import utils as _utils_module, functools

# Silence verbose=True default inside unlearn functions
_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw):
    return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=SEED, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5,
        min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=False, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle_ablation_37', group_name='mu_source',
    )
    d.update(ov)
    return argparse.Namespace(**d)

args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)

# Apply same TEST_MODE shrink as original notebooks
if TEST_MODE:
    def _stratified_subset(ds, fraction, seed=SEED):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_class = collections.defaultdict(list)
        for idx, lbl in enumerate(labels):
            by_class[int(lbl)].append(idx)
        kept = []
        for cls_idx in sorted(by_class):
            cls_pool = by_class[cls_idx]
            rng.shuffle(cls_pool)
            n_keep = max(1, math.ceil(len(cls_pool) * fraction))
            kept.extend(cls_pool[:n_keep])
        sub = torch.utils.data.Subset(ds, kept)
        base_tgt = ds.targets if hasattr(ds, 'targets') else [
            ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_tgt[i] for i in kept]
        return sub
    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: Train={len(dataset_train)}  Test={len(dataset_test)}')
else:
    print(f'Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True,
                 shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, max(1, len(dataset_test))), num_workers=2,
                 pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)

# 3/7 partition  <-- ABLATION
args_part = make_args(unlearn_class=list(FORGET_CLASSES),
                      num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES)
retain_ds, forget_ds = get_retain_forget_partition(args_part, dataset_train, FORGET_CLASSES)
retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
forget_loader = torch.utils.data.DataLoader(forget_ds, **LOADER_KW)

# Full retain loader (no subsampling) — used by recompute_cmf(mean_source='retain')  <-- ABLATION
retain_full_loader = torch.utils.data.DataLoader(
    retain_ds,
    batch_size=min(256, max(1, len(retain_ds))),
    shuffle=False, num_workers=2, pin_memory=True,
)

print(f'\n3/7 Partition statistics:')
print(f'  Forget set  : {len(forget_ds):>6d} samples  ({len(FORGET_CLASSES)} classes: {FORGET_CLASSES})')
print(f'  Retain set  : {len(retain_ds):>6d} samples  ({NUM_CLASSES - len(FORGET_CLASSES)} classes)')
print(f'  Ratio       : {len(forget_ds)/max(1,len(forget_ds)+len(retain_ds))*100:.1f}% forget / '
      f'{len(retain_ds)/max(1,len(forget_ds)+len(retain_ds))*100:.1f}% retain')
print()

# Per-class breakdown
print(f'  {"Class":>5}  {"Name":15}  {"Forget":>8}  {"Retain":>8}')
for c in range(NUM_CLASSES):
    name   = CLASS_LABEL_NAMES[c]
    marker = '  <-- FORGET' if c in FORGET_CLASSES else ''
    n_f = sum(1 for _, lbl in forget_ds if (int(lbl.item()) if torch.is_tensor(lbl) else int(lbl)) == c)
    n_r = sum(1 for _, lbl in retain_ds if (int(lbl.item()) if torch.is_tensor(lbl) else int(lbl)) == c)
    print(f'  {c:>5}  {name:15}  {n_f:>8}  {n_r:>8}{marker}')

print('\nHelpers and data loaded.')

## D. Pre-train

Pre-training uses the **full** training set (same as the original notebook).  
The forget/retain split only applies to retrain baselines and unlearning (Notebook 2).  

If `ORIG_CKPT_DIR` points to an existing pre-train checkpoint from the original Notebook 1,  
it is reused directly — no re-training needed.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR

CKPT_PRETRAIN = f'{CKPT_ROOT}/pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
os.makedirs(os.path.dirname(CKPT_PRETRAIN), exist_ok=True)

# Look for an existing checkpoint from original Notebook 1
_existing = None
if ORIG_CKPT_DIR:
    for _root in [ORIG_CKPT_DIR, f'{ORIG_CKPT_DIR}/checkpoints', f'{ORIG_CKPT_DIR}/checkpoints/ablation_37']:
        _candidate = f'{_root}/pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
        if os.path.exists(_candidate):
            _existing = _candidate
            print(f'Found existing pre-train checkpoint: {_existing}')
            break

if _existing:
    import shutil
    if _existing != CKPT_PRETRAIN:
        shutil.copy2(_existing, CKPT_PRETRAIN)
    args_pt = make_args(unlearn_method='pre_train', num_classes=NUM_CLASSES,
                        class_label_names=CLASS_LABEL_NAMES)
    orig_model = get_model(args_pt, device)
    orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
    print('Loaded from cache — skipping pre-training.')
else:
    print(f'Pre-training {ARCH} on full {DATASET} for up to {PRETRAIN_EPOCHS} epochs ...')
    args_pt = make_args(unlearn_method='pre_train', epochs_or_steps=PRETRAIN_EPOCHS,
                        lr=PRETRAIN_LR, num_classes=NUM_CLASSES,
                        class_label_names=CLASS_LABEL_NAMES, patience=PRETRAIN_PATIENCE)
    orig_model = get_model(args_pt, device)

    total_len = len(dataset_train)
    val_len   = int(total_len * 0.1)
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(total_len, generator=g).tolist()
    tr_loader_pt = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset_train, idx[:total_len - val_len]), **LOADER_KW)
    va_loader_pt = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset_train, idx[total_len - val_len:]), **TEST_KW)

    optimizer = optim.SGD(orig_model.parameters(), lr=PRETRAIN_LR,
                          momentum=0.9, weight_decay=5e-4, nesterov=True)
    _warmup_ms   = min(5, PRETRAIN_EPOCHS)
    warmup_sched = LambdaLR(optimizer, lr_lambda=lambda e: min(1.0, (e+1)/max(1, _warmup_ms)))
    cosine_sched = CosineAnnealingLR(optimizer,
                                     T_max=max(1, PRETRAIN_EPOCHS - _warmup_ms), eta_min=1e-5)
    scheduler    = SequentialLR(optimizer,
                                schedulers=[warmup_sched, cosine_sched],
                                milestones=[_warmup_ms])

    best_acc, no_impr = 0.0, 0
    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        train_one_epoch(args_pt, orig_model, device, tr_loader_pt, optimizer, epoch)
        va, _, _ = test(orig_model, device, va_loader_pt, [], CLASS_LABEL_NAMES,
                        NUM_CLASSES, plot_cm=False, job_name='pretrain', set_name='Val')
        scheduler.step()
        if va > best_acc:
            best_acc, no_impr = va, 0
            torch.save(orig_model.state_dict(), CKPT_PRETRAIN)
            print(f'  epoch {epoch:3d}: val={va:.4f}  saved')
        else:
            no_impr += 1
            if no_impr >= PRETRAIN_PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break
    orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))

orig_model.eval()
print('\n-- Original model test accuracy (full test set) --')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES,
     plot_cm=False, job_name='original', set_name='Test')
print('\n-- Original model accuracy by forget / retain split --')
test(orig_model, device, test_loader, FORGET_CLASSES, CLASS_LABEL_NAMES, NUM_CLASSES,
     plot_cm=False, job_name='original', set_name='Test (3/7 split)')
print(f'\nCheckpoint saved: {CKPT_PRETRAIN}')

## E. Retrain Baseline — 3/7 Retain Set  (Appendix A.5)

Train from scratch on the **retain set only** (classes 3–9, 35 000 samples).  
This is the gold-standard upper bound for the 3/7 ablation.

In [ ]:
num_forget = len(forget_ds)
num_retain = len(retain_ds)

rt_ckpt = f'{CKPT_ROOT}/retrain/{DATASET}_{ARCH}_{_MODE_TAG}/{_FORGET_STR}.pt'
os.makedirs(os.path.dirname(rt_ckpt), exist_ok=True)

print(f'Retrain partition: forget={num_forget} ({len(FORGET_CLASSES)} classes)  retain={num_retain}')

if os.path.exists(rt_ckpt):
    print(f'Loading cached retrain checkpoint: {rt_ckpt}')
    rt_args  = make_args(num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
                         num_retain_samples=num_retain, num_forget_samples=num_forget,
                         unlearn_class=list(FORGET_CLASSES))
    rt_model = get_model(rt_args, device)
    rt_model.load_state_dict(torch.load(rt_ckpt, map_location=device))
else:
    rt_args = make_args(
        unlearn_method='retrain',
        epochs_or_steps=RETRAIN_EPOCHS, lr=PRETRAIN_LR,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        num_retain_samples=num_retain, num_forget_samples=num_forget,
        unlearn_class=list(FORGET_CLASSES),
    )
    rt_model = get_model(rt_args, device)
    rt_opt   = optim.SGD(rt_model.parameters(), lr=PRETRAIN_LR,
                         momentum=0.9, weight_decay=5e-4, nesterov=True)
    print(f'Retraining on retain set for {RETRAIN_EPOCHS} epochs ...')
    rt_model = unlear_func['retrain'](
        args=rt_args, model=rt_model, device=device,
        retain_loader=retain_loader, forget_loader=None,
        train_loader=retain_loader, val_loader=None,
        test_loader=test_loader, optimizer=rt_opt,
        epochs=RETRAIN_EPOCHS, train_dataset=dataset_train,
        val_index=np.arange(len(dataset_train)),
        test_forget_loader=torch.utils.data.DataLoader(dataset_test, **TEST_KW),
    )
    torch.save(rt_model.state_dict(), rt_ckpt)
    print(f'Saved: {rt_ckpt}')

rt_model.eval()
print('\n-- Retrain baseline accuracy (3/7 split) --')
rt_ra, rt_fa, _ = test(rt_model, device, test_loader, FORGET_CLASSES,
                        CLASS_LABEL_NAMES, NUM_CLASSES,
                        plot_cm=False, job_name='retrain', set_name='Test')
print(f'  Retain acc = {rt_ra:.4f}  Forget acc = {rt_fa:.4f}')
RETRAIN_RESULT = dict(forget=_FORGET_STR, retain_acc=rt_ra, forget_acc=rt_fa)
print('\nRetrain baseline done.')

## F. CMF Fine-Tune Encoder — Prepare CMF Starting Point

Fine-tune the original encoder for 1 epoch with the CMF head on the **full** training set  
(same as original Notebook 1 — the CMF head is aligned to all 10 classes).  
This is the starting checkpoint for all CMF-based unlearning methods in Ablation Notebook 2.

In [ ]:
CMF_FT_LR     = 1e-3
CMF_FT_EPOCHS = 1

CKPT_CMF_FT = f'{CKPT_ROOT}/CMF_FT_RemoveFC/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
os.makedirs(os.path.dirname(CKPT_CMF_FT), exist_ok=True)

if os.path.exists(CKPT_CMF_FT):
    print(f'CMF_FT checkpoint already exists: {CKPT_CMF_FT} — skipping training.')
else:
    args_cmf_ft = make_args(
        unlearn_method='CMF_FT_RemoveFC',
        epochs_or_steps=CMF_FT_EPOCHS,
        lr=CMF_FT_LR,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        num_retain_samples=len(dataset_train), num_forget_samples=0,
        unlearn_class=[], remove_FC=True, CMFClassifier=True,
        weight_decay=1e-4,
    )
    cmf_base = get_model(args_cmf_ft, device)
    print('Loading pretrained weights into CMF model ...')
    load_encoder_ckpt_safely(cmf_base, CKPT_PRETRAIN, device=str(device))
    opt_ft = optim.SGD(cmf_base.parameters(), lr=CMF_FT_LR,
                       momentum=0.9, weight_decay=1e-4, nesterov=True)
    print('Running CMF_FT_RemoveFC (1 epoch on full train) ...')
    cmf_ft_model = unlear_func['CMF_FT_RemoveFC'](
        args=args_cmf_ft, model=cmf_base, device=device,
        retain_loader=train_loader, forget_loader=None,
        train_loader=train_loader, val_loader=None,
        test_loader=test_loader, optimizer=opt_ft,
        epochs=CMF_FT_EPOCHS, train_dataset=dataset_train,
        val_index=np.arange(len(dataset_train)),
        test_forget_loader=torch.utils.data.DataLoader(dataset_test, **TEST_KW),
    )
    torch.save(cmf_ft_model.state_dict(), CKPT_CMF_FT)
    print(f'Saved: {CKPT_CMF_FT}')

print('CMF_FT_RemoveFC base checkpoint ready.')

## G. Save ablation_config.json for Ablation Notebook 2

In [ ]:
import pandas as pd

config = dict(
    # Mode
    TEST_MODE=TEST_MODE,
    TEST_FRACTION=TEST_FRACTION,
    _MODE_TAG=_MODE_TAG,
    # Dataset / arch
    DATASET=DATASET,
    ARCH=ARCH,
    IS_VIT=IS_VIT,
    SEED=SEED,
    # Sizes
    NUM_CLASSES=NUM_CLASSES,
    CLASS_LABEL_NAMES=CLASS_LABEL_NAMES,
    TOTAL=_TOTAL[DATASET],
    PER_CLASS=_PER_CLASS[DATASET],
    # 3/7 ablation settings  <-- ABLATION
    FORGET_CLASSES=FORGET_CLASSES,
    FORGET_STR=_FORGET_STR,
    NUM_FORGET=len(forget_ds),
    NUM_RETAIN=len(retain_ds),
    # Training hparams
    PRETRAIN_LR=PRETRAIN_LR,
    PRETRAIN_EPOCHS=PRETRAIN_EPOCHS,
    PRETRAIN_BS=PRETRAIN_BS,
    PRETRAIN_PATIENCE=PRETRAIN_PATIENCE,
    RETRAIN_EPOCHS=RETRAIN_EPOCHS,
    # Checkpoint paths
    CKPT_PRETRAIN=CKPT_PRETRAIN,
    CKPT_CMF_FT=CKPT_CMF_FT,
    CKPT_ROOT=CKPT_ROOT,
    # Retrain result
    RETRAIN_RESULT=RETRAIN_RESULT,
)

config_path = f'{CKPT_ROOT}/ablation_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'ablation_config.json saved: {config_path}')

# Also save retrain results as CSV
rt_csv = f'{CKPT_ROOT}/retrain_results.csv'
pd.DataFrame([RETRAIN_RESULT]).to_csv(rt_csv, index=False)
print(f'Retrain results saved: {rt_csv}')

print('\n' + '='*60)
print('ABLATION NOTEBOOK 1 COMPLETE')
print('='*60)
print('Checkpoints saved under:', CKPT_ROOT)
print()
print('Next steps:')
print('  1. Kaggle -> Output tab -> "Add to Dataset"')
print('     (create a dataset from /kaggle/working/checkpoints/ablation_37/)')
print('  2. Open Ablation Notebook 2 and attach that dataset.')
print('  3. Set CKPT_DATASET_DIR in Ablation Notebook 2 to the dataset mount path.')
print()
print('Summary:')
print(f'  Forget classes : {FORGET_CLASSES} (airplane, automobile, bird)')
print(f'  Forget samples : {len(forget_ds)} ({len(forget_ds)/(len(forget_ds)+len(retain_ds))*100:.0f}%)')
print(f'  Retain samples : {len(retain_ds)} ({len(retain_ds)/(len(forget_ds)+len(retain_ds))*100:.0f}%)')
print(f'  Retrain result : retain={RETRAIN_RESULT["retain_acc"]:.4f}  forget={RETRAIN_RESULT["forget_acc"]:.4f}')